# Лабораторная работа №5

Моделирование сети массового обслуживания с динамическими агентами на примере `RPC`-сервиса для `key-value` хранилища.


## Предметная область

Сеть состоит из трех узлов:

- `RPC Gateway` --- прием и разбор RPC-запроса;
- `Application Logic` --- прикладная логика;
- `Storage + Response` --- работа с хранилищем и отправка ответа.

Каждая заявка проходит все три узла. В каждом узле число активных агентов меняется во времени, поэтому фактическая пропускная способность сети является динамической.


In [ ]:
from collections import deque
from dataclasses import dataclass, field
from heapq import heappop, heappush

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

BASE_SEED = 42
LAMBDA_QPS = 3.2
MU_RPC_QPS = 30.0
MU_RESP_QPS = 40.0
DATABASE_SIZE = 1000
SIMULATION_TIME_SEC = 3000.0
REPLICATIONS = 12
COMMAND_TYPES = ["GET", "SET", "DELETE"]
COMMAND_PROBABILITIES = np.array([0.8, 0.15, 0.05], dtype=float)
NODE_NAMES = ["RPC Gateway", "Application Logic", "Storage + Response"]
BASE_NODE_CONFIGS = [
    {"name": "RPC Gateway", "min_agents": 1, "max_agents": 4, "initial_agents": 2, "connect_rate": 0.40, "disconnect_rate": 0.18},
    {"name": "Application Logic", "min_agents": 1, "max_agents": 3, "initial_agents": 2, "connect_rate": 0.35, "disconnect_rate": 0.24},
    {"name": "Storage + Response", "min_agents": 1, "max_agents": 4, "initial_agents": 2, "connect_rate": 0.30, "disconnect_rate": 0.16},
]

@dataclass
class Request:
    request_id: int
    command: str
    enter_time: float
    stage_times: list[float]
    node_history: list[dict] = field(default_factory=list)

@dataclass
class NodeState:
    name: str
    min_agents: int
    max_agents: int
    initial_agents: int
    connect_rate: float
    disconnect_rate: float
    queue: deque = field(default_factory=deque)
    active_agents: int = 0
    busy_agents: int = 0
    pending_disconnects: int = 0
    area_queue: float = 0.0
    area_active_agents: float = 0.0
    area_busy_agents: float = 0.0
    arrivals: int = 0
    completions: int = 0
    wait_times: list[float] = field(default_factory=list)
    service_times: list[float] = field(default_factory=list)
    connect_events: int = 0
    disconnect_events: int = 0

    def __post_init__(self) -> None:
        self.active_agents = self.initial_agents

def ms(seconds: float) -> float:
    return seconds * 1000.0

def generate_execution_time(command: str, db_size: int, rng: np.random.Generator) -> float:
    means = {"GET": np.log(db_size) * 0.020, "DELETE": np.log(db_size) * 0.024, "SET": db_size * 0.00020}
    return float(rng.exponential(means[command]))

def sample_request_profile(mu_rpc: float, mu_resp: float, db_size: int, rng: np.random.Generator, service_rate_scale: float = 1.0):
    command = str(rng.choice(COMMAND_TYPES, p=COMMAND_PROBABILITIES))
    scale = max(service_rate_scale, 1e-9)
    rpc_time = float(rng.exponential(1.0 / mu_rpc)) / scale
    exec_time = generate_execution_time(command, db_size, rng) / scale
    resp_time = float(rng.exponential(1.0 / mu_resp)) / scale
    stage_times = [rpc_time + 0.10 * exec_time, 0.60 * exec_time, 0.30 * exec_time + resp_time]
    return command, stage_times, {"rpc_time": rpc_time, "exec_time": exec_time, "resp_time": resp_time, "total_service_time": sum(stage_times)}

def estimate_service_model(mu_rpc: float, mu_resp: float, db_size: int, samples: int = 50_000, seed: int = BASE_SEED, service_rate_scale: float = 1.0) -> dict:
    rng = np.random.default_rng(seed)
    stage_matrix = []
    total_values = []
    for _ in range(samples):
        _, stage_times, components = sample_request_profile(mu_rpc, mu_resp, db_size, rng, service_rate_scale)
        stage_matrix.append(stage_times)
        total_values.append(components["total_service_time"])
    stage_array = np.array(stage_matrix, dtype=float)
    total_array = np.array(total_values, dtype=float)
    stage_means = stage_array.mean(axis=0)
    return {
        "mean_total_service_time_sec": float(total_array.mean()),
        "mean_total_service_time_ms": ms(float(total_array.mean())),
        "mu_eff_total_qps": float(1.0 / total_array.mean()),
        "second_moment_total": float((total_array**2).mean()),
        "node_service_time_sec": {name: float(stage_means[idx]) for idx, name in enumerate(NODE_NAMES)},
        "node_service_time_ms": {name: ms(float(stage_means[idx])) for idx, name in enumerate(NODE_NAMES)},
        "node_mu_qps": {name: float(1.0 / stage_means[idx]) for idx, name in enumerate(NODE_NAMES)},
    }

def build_node_configs(alpha_scale: float = 1.0, beta_scale: float = 1.0, max_agents_override: int | None = None) -> list[dict]:
    configs = []
    for cfg in BASE_NODE_CONFIGS:
        max_agents = max_agents_override if max_agents_override is not None else cfg["max_agents"]
        initial_agents = min(cfg["initial_agents"], max_agents)
        initial_agents = max(initial_agents, cfg["min_agents"])
        configs.append(
            {
                "name": cfg["name"],
                "min_agents": cfg["min_agents"],
                "max_agents": max_agents,
                "initial_agents": initial_agents,
                "connect_rate": cfg["connect_rate"] * alpha_scale,
                "disconnect_rate": cfg["disconnect_rate"] * beta_scale,
            }
        )
    return configs

def simulate_network(lambda_qps: float, mu_rpc: float, mu_resp: float, db_size: int, simulation_time: float, seed: int, node_configs: list[dict] | None = None, collect_timeline: bool = False, timeline_horizon: float = 400.0, service_rate_scale: float = 1.0) -> dict:
    rng = np.random.default_rng(seed)
    node_states = [NodeState(**cfg) for cfg in (node_configs or build_node_configs())]
    event_queue = []
    seq = 0
    request_id = 0
    last_event_time = 0.0
    area_network_queue = 0.0
    area_network_system = 0.0
    area_network_active_agents = 0.0
    completed_network_times = []
    requests = {}
    timeline = {"time": [], "network_queue": [], "network_active_agents": [], "network_busy_agents": [], "nodes": {node.name: {"queue": [], "active_agents": [], "busy_agents": []} for node in node_states}}

    def schedule(time_value: float, event_type: str, payload: dict) -> None:
        nonlocal seq
        seq += 1
        heappush(event_queue, (float(time_value), seq, event_type, payload))

    def total_queue_length() -> int:
        return sum(len(node.queue) for node in node_states)

    def total_busy_agents() -> int:
        return sum(node.busy_agents for node in node_states)

    def total_active_agents() -> int:
        return sum(node.active_agents for node in node_states)

    def record_snapshot(current_time: float) -> None:
        if not collect_timeline or current_time > timeline_horizon:
            return
        timeline["time"].append(float(current_time))
        timeline["network_queue"].append(total_queue_length())
        timeline["network_active_agents"].append(total_active_agents())
        timeline["network_busy_agents"].append(total_busy_agents())
        for node in node_states:
            timeline["nodes"][node.name]["queue"].append(len(node.queue))
            timeline["nodes"][node.name]["active_agents"].append(node.active_agents)
            timeline["nodes"][node.name]["busy_agents"].append(node.busy_agents)

    def update_areas(current_time: float) -> None:
        nonlocal last_event_time, area_network_queue, area_network_system, area_network_active_agents
        dt = current_time - last_event_time
        if dt <= 0:
            return
        for node in node_states:
            node.area_queue += len(node.queue) * dt
            node.area_active_agents += node.active_agents * dt
            node.area_busy_agents += node.busy_agents * dt
        area_network_queue += total_queue_length() * dt
        area_network_active_agents += total_active_agents() * dt
        area_network_system += (total_queue_length() + total_busy_agents()) * dt
        last_event_time = current_time

    def schedule_connect_disconnect_events() -> None:
        for node_idx, node in enumerate(node_states):
            if node.connect_rate > 0:
                schedule(rng.exponential(1.0 / node.connect_rate), "connect", {"node_idx": node_idx})
            if node.disconnect_rate > 0:
                schedule(rng.exponential(1.0 / node.disconnect_rate), "disconnect", {"node_idx": node_idx})

    def schedule_next_external_arrival(base_time: float) -> None:
        if lambda_qps > 0:
            schedule(base_time + rng.exponential(1.0 / lambda_qps), "external_arrival", {})

    def maybe_start_service(node_idx: int, current_time: float) -> None:
        node = node_states[node_idx]
        effective_capacity = max(0, node.active_agents - node.pending_disconnects)
        while node.queue and node.busy_agents < effective_capacity:
            req_id, node_arrival_time = node.queue.popleft()
            req = requests[req_id]
            node.busy_agents += 1
            wait_time = current_time - node_arrival_time
            service_time = req.stage_times[node_idx]
            node.wait_times.append(wait_time)
            node.service_times.append(service_time)
            req.node_history.append({"node_name": node.name, "queue_enter_time": node_arrival_time, "service_start_time": current_time, "service_end_time": current_time + service_time, "wait_time": wait_time, "service_time": service_time})
            schedule(current_time + service_time, "service_completion", {"node_idx": node_idx, "request_id": req_id})
            effective_capacity = max(0, node.active_agents - node.pending_disconnects)

    def arrive_to_node(node_idx: int, req_id: int, current_time: float) -> None:
        node = node_states[node_idx]
        node.arrivals += 1
        node.queue.append((req_id, current_time))
        maybe_start_service(node_idx, current_time)

    def process_external_arrival(current_time: float) -> None:
        nonlocal request_id
        request_id += 1
        command, stage_times, _ = sample_request_profile(mu_rpc, mu_resp, db_size, rng, service_rate_scale)
        requests[request_id] = Request(request_id=request_id, command=command, enter_time=current_time, stage_times=stage_times)
        arrive_to_node(0, request_id, current_time)
        schedule_next_external_arrival(current_time)

    def process_service_completion(node_idx: int, req_id: int, current_time: float) -> None:
        node = node_states[node_idx]
        req = requests[req_id]
        node.busy_agents = max(0, node.busy_agents - 1)
        node.completions += 1
        if node.pending_disconnects > 0 and node.active_agents > node.min_agents:
            node.active_agents -= 1
            node.pending_disconnects -= 1
        if node_idx < len(node_states) - 1:
            arrive_to_node(node_idx + 1, req_id, current_time)
        else:
            completed_network_times.append(current_time - req.enter_time)
        maybe_start_service(node_idx, current_time)

    def process_connect(node_idx: int, current_time: float) -> None:
        node = node_states[node_idx]
        if node.active_agents < node.max_agents:
            node.active_agents += 1
            node.connect_events += 1
            maybe_start_service(node_idx, current_time)
        if node.connect_rate > 0:
            schedule(current_time + rng.exponential(1.0 / node.connect_rate), "connect", {"node_idx": node_idx})

    def process_disconnect(node_idx: int, current_time: float) -> None:
        node = node_states[node_idx]
        removable_agents = node.active_agents - node.pending_disconnects
        if removable_agents > node.min_agents:
            idle_agents = node.active_agents - node.busy_agents - node.pending_disconnects
            if idle_agents > 0:
                node.active_agents -= 1
            else:
                node.pending_disconnects += 1
            node.disconnect_events += 1
        if node.disconnect_rate > 0:
            schedule(current_time + rng.exponential(1.0 / node.disconnect_rate), "disconnect", {"node_idx": node_idx})

    schedule_connect_disconnect_events()
    schedule_next_external_arrival(0.0)
    record_snapshot(0.0)
    while event_queue:
        current_time, _, event_type, payload = heappop(event_queue)
        capped_time = min(current_time, simulation_time)
        update_areas(capped_time)
        if current_time > simulation_time:
            break
        if event_type == "external_arrival":
            process_external_arrival(current_time)
        elif event_type == "service_completion":
            process_service_completion(payload["node_idx"], payload["request_id"], current_time)
        elif event_type == "connect":
            process_connect(payload["node_idx"], current_time)
        elif event_type == "disconnect":
            process_disconnect(payload["node_idx"], current_time)
        record_snapshot(current_time)
    update_areas(simulation_time)

    node_results = []
    for node in node_states:
        mean_wait = float(np.mean(node.wait_times)) if node.wait_times else 0.0
        mean_service = float(np.mean(node.service_times)) if node.service_times else 0.0
        node_results.append({"node_name": node.name, "mean_wait_ms": ms(mean_wait), "mean_service_ms": ms(mean_service), "mean_queue_length": node.area_queue / simulation_time, "mean_active_agents_node": node.area_active_agents / simulation_time, "utilization_node": node.area_busy_agents / node.area_active_agents if node.area_active_agents > 0 else 0.0, "arrivals": node.arrivals, "completions": node.completions, "connect_events": node.connect_events, "disconnect_events": node.disconnect_events})

    network_result = {
        "mean_network_time_ms": ms(float(np.mean(completed_network_times))) if completed_network_times else 0.0,
        "mean_network_queue": area_network_queue / simulation_time,
        "mean_network_system": area_network_system / simulation_time,
        "mean_active_agents_total": area_network_active_agents / simulation_time,
        "completed_requests": len(completed_network_times),
        "loss_probability": 0.0,
        "node_waits_ms": {item["node_name"]: item["mean_wait_ms"] for item in node_results},
        "node_queue_lengths": {item["node_name"]: item["mean_queue_length"] for item in node_results},
        "node_active_agents": {item["node_name"]: item["mean_active_agents_node"] for item in node_results},
        "node_utilization": {item["node_name"]: item["utilization_node"] for item in node_results},
    }
    return {"network": network_result, "nodes": node_results, "timeline": timeline if collect_timeline else None}

def aggregate_network_replications(lambda_qps: float, mu_rpc: float, mu_resp: float, db_size: int, simulation_time: float, replications: int, base_seed: int = BASE_SEED, node_configs: list[dict] | None = None, service_rate_scale: float = 1.0) -> dict:
    runs = [simulate_network(lambda_qps, mu_rpc, mu_resp, db_size, simulation_time, base_seed + idx, node_configs=node_configs, service_rate_scale=service_rate_scale) for idx in range(replications)]
    network_metrics = {}
    scalar_fields = ["mean_network_time_ms", "mean_network_queue", "mean_network_system", "mean_active_agents_total", "completed_requests", "loss_probability"]
    for field_name in scalar_fields:
        values = np.array([run["network"][field_name] for run in runs], dtype=float)
        network_metrics[field_name] = float(values.mean())
        network_metrics[f"{field_name}_std"] = float(values.std(ddof=1)) if len(values) > 1 else 0.0
    for nested_name in ["node_waits_ms", "node_queue_lengths", "node_active_agents", "node_utilization"]:
        network_metrics[nested_name] = {node_name: float(np.mean([run["network"][nested_name][node_name] for run in runs])) for node_name in NODE_NAMES}
    node_results = []
    template_keys = ["mean_wait_ms", "mean_service_ms", "mean_queue_length", "mean_active_agents_node", "utilization_node", "arrivals", "completions", "connect_events", "disconnect_events"]
    for node_name in NODE_NAMES:
        aggregated = {"node_name": node_name}
        for key in template_keys:
            aggregated[key] = float(np.mean([next(node[key] for node in run["nodes"] if node["node_name"] == node_name) for run in runs]))
        node_results.append(aggregated)
    return {"network": network_metrics, "nodes": node_results, "replications": replications}

def build_summary() -> dict:
    service_model = estimate_service_model(MU_RPC_QPS, MU_RESP_QPS, DATABASE_SIZE)
    base_configs = build_node_configs()
    base_results = aggregate_network_replications(LAMBDA_QPS, MU_RPC_QPS, MU_RESP_QPS, DATABASE_SIZE, SIMULATION_TIME_SEC, REPLICATIONS, node_configs=base_configs)
    alpha_rows = []
    for scale in [0.5, 0.75, 1.0, 1.25, 1.5]:
        result = aggregate_network_replications(LAMBDA_QPS, MU_RPC_QPS, MU_RESP_QPS, DATABASE_SIZE, SIMULATION_TIME_SEC, REPLICATIONS, base_seed=BASE_SEED + int(scale * 100), node_configs=build_node_configs(alpha_scale=scale))
        alpha_rows.append({"alpha_scale": scale, "mean_network_time_ms": result["network"]["mean_network_time_ms"], "mean_network_queue": result["network"]["mean_network_queue"], "mean_active_agents_total": result["network"]["mean_active_agents_total"], "app_logic_wait_ms": result["network"]["node_waits_ms"]["Application Logic"]})
    beta_rows = []
    for scale in [0.5, 0.75, 1.0, 1.25, 1.5]:
        result = aggregate_network_replications(LAMBDA_QPS, MU_RPC_QPS, MU_RESP_QPS, DATABASE_SIZE, SIMULATION_TIME_SEC, REPLICATIONS, base_seed=BASE_SEED + 500 + int(scale * 100), node_configs=build_node_configs(beta_scale=scale))
        beta_rows.append({"beta_scale": scale, "mean_network_time_ms": result["network"]["mean_network_time_ms"], "mean_network_queue": result["network"]["mean_network_queue"], "mean_active_agents_total": result["network"]["mean_active_agents_total"], "app_logic_queue": result["network"]["node_queue_lengths"]["Application Logic"]})
    lambda_rows = []
    for lambda_value in [2.0, 2.8, 3.6, 4.4, 5.2]:
        result = aggregate_network_replications(lambda_value, MU_RPC_QPS, MU_RESP_QPS, DATABASE_SIZE, SIMULATION_TIME_SEC, REPLICATIONS, base_seed=BASE_SEED + 1000 + int(lambda_value * 100), node_configs=base_configs)
        lambda_rows.append({"lambda_qps": lambda_value, "mean_network_time_ms": result["network"]["mean_network_time_ms"], "mean_network_queue": result["network"]["mean_network_queue"], "mean_active_agents_total": result["network"]["mean_active_agents_total"], "app_logic_wait_ms": result["network"]["node_waits_ms"]["Application Logic"]})
    agent_limit_rows = []
    for max_agents in [2, 3, 4, 5]:
        result = aggregate_network_replications(LAMBDA_QPS, MU_RPC_QPS, MU_RESP_QPS, DATABASE_SIZE, SIMULATION_TIME_SEC, REPLICATIONS, base_seed=BASE_SEED + 1500 + max_agents, node_configs=build_node_configs(max_agents_override=max_agents))
        agent_limit_rows.append({"max_agents": max_agents, "mean_network_time_ms": result["network"]["mean_network_time_ms"], "mean_network_queue": result["network"]["mean_network_queue"], "mean_active_agents_total": result["network"]["mean_active_agents_total"], "app_logic_utilization": result["network"]["node_utilization"]["Application Logic"]})
    scenario_defs = [("Стабильная сеть", 2.6, 1.20, 0.70, None, 1.0), ("Базовый режим", LAMBDA_QPS, 1.00, 1.00, None, 1.0), ("Высокая нестабильность агентов", LAMBDA_QPS, 0.85, 1.35, None, 1.0), ("Высокая нагрузка", 4.8, 1.00, 1.00, None, 1.0), ("Улучшенный режим", 4.8, 1.25, 0.85, 5, 1.1)]
    scenario_rows = []
    for idx, (label, lambda_value, alpha_scale, beta_scale, max_override, service_scale) in enumerate(scenario_defs):
        result = aggregate_network_replications(lambda_value, MU_RPC_QPS, MU_RESP_QPS, DATABASE_SIZE, SIMULATION_TIME_SEC, REPLICATIONS, base_seed=BASE_SEED + 2000 + idx * 100, node_configs=build_node_configs(alpha_scale=alpha_scale, beta_scale=beta_scale, max_agents_override=max_override), service_rate_scale=service_scale)
        scenario_rows.append({"scenario": label, "lambda_qps": lambda_value, "alpha_scale": alpha_scale, "beta_scale": beta_scale, "mean_network_time_ms": result["network"]["mean_network_time_ms"], "mean_network_queue": result["network"]["mean_network_queue"], "mean_active_agents_total": result["network"]["mean_active_agents_total"], "app_logic_wait_ms": result["network"]["node_waits_ms"]["Application Logic"]})
    timeline_run = simulate_network(LAMBDA_QPS, MU_RPC_QPS, MU_RESP_QPS, DATABASE_SIZE, 400.0, BASE_SEED + 777, node_configs=base_configs, collect_timeline=True, timeline_horizon=400.0)
    return {"service_model": service_model, "base_results": base_results, "alpha_experiments": alpha_rows, "beta_experiments": beta_rows, "lambda_experiments": lambda_rows, "agent_limit_experiments": agent_limit_rows, "scenario_results": scenario_rows, "timeline_results": timeline_run["timeline"]}

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


## Параметры модели


In [ ]:
pd.DataFrame([
    {"Параметр": "λ", "Значение": f"{LAMBDA_QPS} qps"},
    {"Параметр": "μ_rpc", "Значение": f"{MU_RPC_QPS} qps"},
    {"Параметр": "μ_resp", "Значение": f"{MU_RESP_QPS} qps"},
    {"Параметр": "Размер БД", "Значение": DATABASE_SIZE},
    {"Параметр": "Время моделирования", "Значение": f"{SIMULATION_TIME_SEC} с"},
    {"Параметр": "Число прогонов", "Значение": REPLICATIONS},
])


## Оценка сервисной модели


In [ ]:
service_model = estimate_service_model(MU_RPC_QPS, MU_RESP_QPS, DATABASE_SIZE)
node_service_summary = pd.DataFrame([
    {"Узел": node_name, "E[S_i], мс": service_model["node_service_time_ms"][node_name], "mu_i, qps": service_model["node_mu_qps"][node_name]}
    for node_name in service_model["node_service_time_ms"]
])
display(node_service_summary.round(3))
pd.Series({
    "E[S_total], с": service_model["mean_total_service_time_sec"],
    "E[S_total], мс": service_model["mean_total_service_time_ms"],
    "mu_eff_total, qps": service_model["mu_eff_total_qps"],
}).round(4)


## Математическая постановка

Для каждого узла вводятся:

- `μ_i` --- интенсивность обслуживания одного агента;
- `m_i(t)` --- текущее число активных агентов;
- `α_i` --- интенсивность подключений;
- `β_i` --- интенсивность отключений.

Текущая пропускная способность узла:

$$
\mu_i^{total}(t) = m_i(t) \mu_i
$$

Если средний входящий поток на узел длительно превышает его среднюю пропускную способность, в узле начинает расти очередь и он становится узким местом сети.


In [ ]:
summary = build_summary()


## Базовый эксперимент


In [ ]:
base_network = pd.DataFrame([summary["base_results"]["network"]]).drop(columns=["node_waits_ms", "node_queue_lengths", "node_active_agents", "node_utilization"])
base_nodes = pd.DataFrame(summary["base_results"]["nodes"])
print("Метрики сети:")
display(base_network.round(3))
print("Метрики узлов:")
display(base_nodes.round(3))


## Эксперименты


In [ ]:
alpha_experiments = pd.DataFrame(summary["alpha_experiments"])
beta_experiments = pd.DataFrame(summary["beta_experiments"])
lambda_experiments = pd.DataFrame(summary["lambda_experiments"])
agent_limit_experiments = pd.DataFrame(summary["agent_limit_experiments"])

print("Влияние интенсивности подключений:")
display(alpha_experiments.round(3))
print("Влияние интенсивности отключений:")
display(beta_experiments.round(3))
print("Влияние входной интенсивности:")
display(lambda_experiments.round(3))
print("Влияние верхнего предела числа агентов:")
display(agent_limit_experiments.round(3))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.lineplot(data=alpha_experiments, x="alpha_scale", y="mean_network_time_ms", marker="o", ax=axes[0, 0])
axes[0, 0].set_title("Среднее время в сети от alpha")
axes[0, 0].set_xlabel("Масштаб alpha")
axes[0, 0].set_ylabel("Время в сети, мс")

sns.lineplot(data=beta_experiments, x="beta_scale", y="mean_network_queue", marker="o", ax=axes[0, 1])
axes[0, 1].set_title("Средняя длина очереди от beta")
axes[0, 1].set_xlabel("Масштаб beta")
axes[0, 1].set_ylabel("Средняя очередь")

sns.lineplot(data=lambda_experiments, x="lambda_qps", y="mean_network_time_ms", marker="o", ax=axes[1, 0])
axes[1, 0].set_title("Среднее время в сети от lambda")
axes[1, 0].set_xlabel("lambda, qps")
axes[1, 0].set_ylabel("Время в сети, мс")

sns.lineplot(data=agent_limit_experiments, x="max_agents", y="mean_active_agents_total", marker="o", ax=axes[1, 1])
axes[1, 1].set_title("Среднее число активных агентов")
axes[1, 1].set_xlabel("Максимум агентов в узле")
axes[1, 1].set_ylabel("Активные агенты")

plt.tight_layout()
plt.show()


## Временные графики


In [ ]:
timeline = summary["timeline_results"]
time_grid = timeline["time"]
node_names = list(timeline["nodes"].keys())

fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
for node_name in node_names:
    axes[0].plot(time_grid, timeline["nodes"][node_name]["active_agents"], label=node_name, linewidth=2)
    axes[1].plot(time_grid, timeline["nodes"][node_name]["queue"], label=node_name, linewidth=2)
    axes[2].plot(time_grid, timeline["nodes"][node_name]["busy_agents"], label=node_name, linewidth=2)

axes[0].set_title("Число активных агентов во времени")
axes[0].set_ylabel("Активные агенты")
axes[1].set_title("Длина очереди по узлам")
axes[1].set_ylabel("Очередь")
axes[2].set_title("Число занятых агентов")
axes[2].set_ylabel("Занятые агенты")
axes[2].set_xlabel("Время, с")
for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()


## Сценарный анализ


In [ ]:
scenario_results = pd.DataFrame(summary["scenario_results"])
display(scenario_results.round(3))


## Выводы

1. Динамика агентов напрямую влияет на текущую пропускную способность узлов.
2. В базовом режиме узел `Application Logic` чаще всего становится узким местом.
3. Рост интенсивности отключений увеличивает очереди и среднее время пребывания заявки в сети.
4. Рост интенсивности подключений и увеличение лимита числа агентов стабилизируют сеть.
